# Feature Engineering for Customer Churn Prediction 📡

**Project Objective:** To highlight the impact of feature engineering on predictive performance by comparing two models:  
- a **baseline model** trained on raw features, and  
- an **enhanced model** built with carefully engineered features.  

The aim is to improve churn prediction accuracy for a telecommunications company while showcasing practical feature engineering techniques.

### Core Concepts:
1. **Why Feature Engineering Matters:** Understanding how better features often drive bigger performance gains than complex models.  
2. **Data Cleaning in Practice:** Fixing data type mismatches, missing values, and inconsistencies common in real-world datasets.  
3. **Feature Creation Strategies:**
   - **Binning/Discretization:** Transforming continuous variables into meaningful categories (e.g., tenure buckets).  
   - **Feature Aggregation:** Combining services or behaviors into single, interpretable metrics (e.g., total number of subscribed services).  
   - **Category Simplification:** Reducing complexity for improved model interpretability.  
4. **Pipeline Construction:** Leveraging Scikit-Learn’s `ColumnTransformer` and `Pipeline` for reproducible preprocessing workflows.  
5. **Model Evaluation:** Measuring the performance lift from engineered features compared to raw data models.  


### Step 1: Setup - Importing Libraries and Loading Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, accuracy_score, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Set plot style
sns.set_style('whitegrid')

In [2]:
!git clone "https://github.com/GeeksforgeeksDS/21-Days-21-Projects-Dataset"

Cloning into '21-Days-21-Projects-Dataset'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 22 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 1.40 MiB | 4.33 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [3]:
# Load the dataset from the user-provided file
df = pd.read_csv('/content/21-Days-21-Projects-Dataset/Datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("Dataset loaded successfully.")
print(f"Data shape: {df.shape}")
df.head()

Dataset loaded successfully.
Data shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Step 2: Data Cleaning and Initial Preparation
Real-world data is often messy. We need to handle inconsistencies before we can do any analysis or modeling.

In [4]:
# Drop ID column and target separation
df.drop('customerID', axis=1, inplace=True)
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop('Churn', axis=1)

In [5]:
# ------------------------------
# 1. Custom Feature Engineer
# ------------------------------
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, drop_raw_tenure=True):
        self.drop_raw_tenure = drop_raw_tenure

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Convert TotalCharges to numeric
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        X['TotalCharges'].fillna(X['TotalCharges'].median(), inplace=True)

        # tenure groups
        X['tenure_group'] = pd.cut(
            X['tenure'],
            bins=[0, 12, 24, 48, 60, 72],
            labels=['0-12', '13-24', '25-48', '49-60', '61-72']
        )

        # Avg Monthly Charges
        X['AvgMonthlyCharges'] = X['TotalCharges'] / (X['tenure'] + 1)

        # Num Services (count of "Yes")
        service_cols = [
            'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
        ]
        X['NumServices'] = (X[service_cols] == 'Yes').sum(axis=1)

        # Optionally drop raw tenure
        if self.drop_raw_tenure and 'tenure' in X.columns:
            X = X.drop('tenure', axis=1)

        return X


# ------------------------------
# 2. Define Column Groups
# ------------------------------
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
multi_cat_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod', 'tenure_group'
]
num_cols = ['MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharges', 'NumServices', 'SeniorCitizen']
# 🔑 note: 'tenure' removed because drop_raw_tenure=True

# ------------------------------
# 3. Column Transformer
# ------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', OneHotEncoder(drop='if_binary'), binary_cols),
        ('categorical', OneHotEncoder(drop='first'), multi_cat_cols),
        ('numeric', StandardScaler(), num_cols)
    ],
    remainder='drop'
)

# ------------------------------
# 4. Full Pipeline
# ------------------------------
pipeline = Pipeline(steps=[
    ('feature_engineering', FeatureEngineer(drop_raw_tenure=True)),
    ('preprocessing', preprocessor)
])

# ------------------------------
# 5. Train-Test Split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit and transform
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

print("Processed Train Shape:", X_train_processed.shape)
print("Processed Test Shape:", X_test_processed.shape)


Processed Train Shape: (5634, 36)
Processed Test Shape: (1409, 36)


/tmp/ipython-input-1800773818.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['TotalCharges'].fillna(X['TotalCharges'].median(), inplace=True)
/tmp/ipython-input-1800773818.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in

### 3. Model 1 - Baseline Performance (Without Feature Engineering)
First, we'll build a model using only the original, cleaned features. This will serve as our benchmark to see if our feature engineering efforts actually help.

In [6]:
# Define features (X) and target (y)
X_base = df.drop('Churn', axis=1)
y_base = df['Churn']

# Identify categorical and numerical features
numerical_features_base = X_base.select_dtypes(include=np.number).columns.tolist()
categorical_features_base = X_base.select_dtypes(include=['object']).columns.tolist()

# Create the preprocessing pipeline
preprocessor_base = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features_base),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_base)])

# Split data
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(X_base, y_base, test_size=0.2, random_state=42, stratify=y_base)

# Create the full pipeline with a classifier
baseline_model = Pipeline(steps=[('preprocessor', preprocessor_base),
                                 ('classifier', LogisticRegression(random_state=42, max_iter=1000))])

# Train and evaluate the baseline model
baseline_model.fit(X_train_base, y_train_base)
y_pred_base = baseline_model.predict(X_test_base)

print("--- Baseline Model Performance ---")
print(classification_report(y_test_base, y_pred_base))

--- Baseline Model Performance ---
              precision    recall  f1-score   support

          No       0.84      0.89      0.86      1035
         Yes       0.63      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409



### 4. Model 2 - Performance with Engineered Features
Now, we'll build a new model using our enriched dataset and see if performance improves.

In [7]:
# ------------------------------
# Logistic Regression Pipeline
# ------------------------------
log_reg_model = Pipeline(steps=[
    ('feature_engineering', FeatureEngineer(drop_raw_tenure=True)),
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

log_reg_model.fit(X_train, y_train)
y_pred_log = log_reg_model.predict(X_test)

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

# ------------------------------
# Random Forest Pipeline
# ------------------------------
rf_model = Pipeline(steps=[
    ('feature_engineering', FeatureEngineer(drop_raw_tenure=True)),
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, class_weight="balanced"))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\n=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


/tmp/ipython-input-1800773818.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['TotalCharges'].fillna(X['TotalCharges'].median(), inplace=True)
/tmp/ipython-input-1800773818.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in

=== Logistic Regression ===
Accuracy: 0.7998580553584103
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.66      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409


=== Random Forest ===
Accuracy: 0.7643718949609652
              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.74      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



/tmp/ipython-input-1800773818.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['TotalCharges'].fillna(X['TotalCharges'].median(), inplace=True)


### 5. Performance with top 10 features

In [9]:
from sklearn.feature_selection import SelectFromModel

# ------------------------------
# 1. Feature Engineering + Preprocessing
# ------------------------------
X_train_fe = FeatureEngineer(drop_raw_tenure=True).fit_transform(X_train)
X_test_fe  = FeatureEngineer(drop_raw_tenure=True).transform(X_test)

X_train_proc = preprocessor.fit_transform(X_train_fe)
X_test_proc  = preprocessor.transform(X_test_fe)

feature_names = preprocessor.get_feature_names_out()

# ------------------------------
# 2. Random Forest for feature selection
# ------------------------------
rf_selector = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_selector.fit(X_train_proc, y_train)

# Threshold for top 10 features
importances = rf_selector.feature_importances_
threshold = np.sort(importances)[-10]  # 10th largest

selector = SelectFromModel(estimator=rf_selector, threshold=threshold, prefit=True)

# Select top 10 features
X_train_selected = selector.transform(X_train_proc)
X_test_selected  = selector.transform(X_test_proc)
selected_features = np.array(feature_names)[selector.get_support()]

print("Top 10 selected features:", selected_features)
print(f"Original training shape: {X_train_proc.shape}, Selected shape: {X_train_selected.shape}")

# ------------------------------
# 3. Train Logistic Regression
# ------------------------------
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_selected, y_train)
y_pred_log = log_reg.predict(X_test_selected)

print("\n=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

# ------------------------------
# 4. Train Random Forest
# ------------------------------
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
rf_model.fit(X_train_selected, y_train)
y_pred_rf = rf_model.predict(X_test_selected)

print("\n=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


/tmp/ipython-input-1800773818.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['TotalCharges'].fillna(X['TotalCharges'].median(), inplace=True)
/tmp/ipython-input-1800773818.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in

Top 10 selected features: ['binary__gender_Male' 'categorical__InternetService_Fiber optic'
 'categorical__OnlineSecurity_Yes' 'categorical__Contract_One year'
 'categorical__Contract_Two year'
 'categorical__PaymentMethod_Electronic check' 'numeric__MonthlyCharges'
 'numeric__TotalCharges' 'numeric__AvgMonthlyCharges'
 'numeric__NumServices']
Original training shape: (5634, 36), Selected shape: (5634, 10)

=== Logistic Regression ===
Accuracy: 0.7920511000709723
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409


=== Random Forest ===
Accuracy: 0.7679205110007097
              precision    recall  f1-score   support

           0       0.89      0.78      0.83      1035
           1       0.55      0.73      0.62 

### 6. Compare model performance

In [17]:
# Function to extract F1-score for the positive class (Churn)
def f1_score_for_churn(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)

    # If target is binary, select the label with higher value or 'Yes' if string
    if set(y_true) <= {'0', '1'} or set(y_true) <= {0, 1}:
        pos_label = '1' if '1' in report else 1
    elif 'Yes' in report:
        pos_label = 'Yes'
    else:
        # fallback: pick the class with lower support? or first key
        pos_label = list(report.keys())[0]

    return report[pos_label]['f1-score']

# ------------------------------
# Classification Reports
# ------------------------------
print("--- Baseline Model Performance (Logistic Regression) ---")
print(classification_report(y_test_base, y_pred_base))

print("\n--- Enhanced Model Performance (Engineered Features, Logistic Regression) ---")
print(classification_report(y_test, y_pred_log))

print("\n--- Top 10 Selected Features Performance (Logistic Regression) ---")
print(classification_report(y_test, y_pred_log))

print("\n--- Top 10 Selected Features Performance (Random Forest) ---")
print(classification_report(y_test, y_pred_rf))

# ------------------------------
# Performance Summary
# ------------------------------
print("\n--- Performance Summary (Accuracy & Churn F1-score) ---")
print("Model / Dataset      | Accuracy | F1-Score (Churn)")
print("---------------------|---------|----------------")

# Baseline
acc_base = accuracy_score(y_test_base, y_pred_base)
f1_base = f1_score_for_churn(y_test_base, y_pred_base)
print(f"Baseline (LR)        | {acc_base:.2f}    | {f1_base:.2f}")

# Enhanced
acc_eng = accuracy_score(y_test, y_pred_log)
f1_eng = f1_score_for_churn(y_test, y_pred_log)
print(f"Enhanced (LR)        | {acc_eng:.2f}    | {f1_eng:.2f}")

# Top 10 (Logistic Regression)
acc_top10_lr = accuracy_score(y_test, y_pred_log)
f1_top10_lr = f1_score_for_churn(y_test, y_pred_log)
print(f"Top 10 Features (LR) | {acc_top10_lr:.2f}    | {f1_top10_lr:.2f}")

# Top 10 (Random Forest)
acc_top10_rf = accuracy_score(y_test, y_pred_rf)
f1_top10_rf = f1_score_for_churn(y_test, y_pred_rf)
print(f"Top 10 Features (RF) | {acc_top10_rf:.2f}    | {f1_top10_rf:.2f}")


--- Baseline Model Performance (Logistic Regression) ---
              precision    recall  f1-score   support

          No       0.84      0.89      0.86      1035
         Yes       0.63      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409


--- Enhanced Model Performance (Engineered Features, Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409


--- Top 10 Selected Features Performance (Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49

### 7. Analysis

## Approaches Explored

### 1. Baseline Model (Raw Features)
- **Description:**  
  Used the dataset as-is with minimal preprocessing.  
  Numeric features were standardized, categorical features were one-hot encoded.  
- **Pipeline:**  
  - `ColumnTransformer` for preprocessing  
  - `LogisticRegression` classifier  
- **Purpose:**  
  To establish a performance baseline without any feature engineering.  
- **Performance Highlights:**  
  - Accuracy: 0.79  
  - F1-score (Churn): 0.58  
- **Insights:**  
  The model is moderately accurate but struggles with detecting churn (minority class).

### 2. Enhanced Features Model (Feature Engineering)
- **Description:**  
  Engineered new features to improve predictive power:  
  - **`tenure_group`**: Binned tenure into categories  
  - **`AvgMonthlyCharges`**: Average monthly charges per customer  
  - **`NumServices`**: Count of active services per customer  
- **Pipeline:**  
  - `FeatureEngineer` transformer → `ColumnTransformer` → `LogisticRegression` / `RandomForestClassifier`  
- **Purpose:**  
  To see if additional features improve model performance over the baseline.  
- **Performance Highlights (LR):**  
  - Accuracy: 0.79  
  - F1-score (Churn): 0.56  
- **Insights:**  
  Feature engineering alone did not significantly improve Logistic Regression performance.

### 3. Top 10 Selected Features Model (Feature Selection)
- **Description:**  
  Selected the **top 10 most important features** using:  
  - `RandomForestClassifier` for feature importance  
  - `SelectFromModel` with threshold set to the 10th highest importance  
- **Pipeline:**  
  - `FeatureEngineer` → `ColumnTransformer` → Top 10 features → `LogisticRegression` / `RandomForestClassifier`  
- **Purpose:**  
  To reduce feature space and focus on the most predictive variables.  
- **Performance Highlights:**  
  - Logistic Regression: Accuracy 0.79, F1-score 0.56  
  - Random Forest: Accuracy 0.77, F1-score 0.62  
- **Insights:**  
  - Logistic Regression showed no change.  
  - Random Forest improved F1-score for churn, highlighting the benefits of feature selection for ensemble models.

### Summary Table of Model Performance

| Model / Dataset            | Accuracy | F1-Score (Churn) |
|----------------------------|---------|-----------------|
| Baseline (LR)              | 0.79    | 0.58            |
| Enhanced (LR)              | 0.79    | 0.56            |
| Top 10 Features (LR)       | 0.79    | 0.56            |
| Top 10 Features (RF)       | 0.77    | 0.62            |

**Key Takeaways:**
- **Baseline LR vs. Enhanced LR**: Feature engineering slightly decreased F1-score for the churn class, though overall accuracy remained the same.
- **Top 10 Features LR**: Logistic Regression did not benefit significantly from feature selection alone.
- **Top 10 Features RF**: Random Forest improved F1-score for churn (minority class), even with slightly lower overall accuracy.  
  This demonstrates that ensemble methods can better leverage selected features to handle class imbalance.

- Feature engineering alone may not always improve Logistic Regression performance.  
- Feature selection can improve minority class detection, especially with ensemble models like Random Forest.  
- Random Forest is more robust to imbalanced classes and leverages important features effectively.
